Here is the final, unified code block. It automatically handles the logic of checking your Google Drive first. If the vector database is already saved there, it loads it instantly. If it isn't there yet, it builds the database using your GPU and then saves it for future use.

This single cell handles the complete workflow:

In [ ]:
!pip install ragas rouge-score bert-score nltk pandas tqdm langchain-google-genai

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.6 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=1

In [ ]:
# 1. Added 'langchain-classic' to handle the new v1.0 architecture
!pip install langchain langchain-classic langchain-text-splitters langchain-google-genai langchain-community langchain-huggingface sentence-transformers faiss-cpu

# 2. Import dependencies (Notice the 'langchain_classic' changes here)
import os
import pandas as pd
import torch
from getpass import getpass
from google.colab import drive
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# 3. Mount Google Drive
drive.mount('/content/drive')

# 4. Define File Paths
csv_file_path = '/content/drive/MyDrive/bangladesh_legal_acts_sections.csv'
db_save_path = '/content/drive/MyDrive/LawBot_FAISS_DB'

# 5. Initialize Embeddings Model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Hardware accelerator set to: {device.upper()}")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': device},
    encode_kwargs={'batch_size': 256}
)

# 6. Smart Vector Database Logic
if os.path.exists(db_save_path):
    print("Found existing vector database in Drive! Loading instantly...")
    vectorstore = FAISS.load_local(
        folder_path=db_save_path,
        embeddings=embeddings,
        allow_dangerous_deserialization=True
    )
    print("Database successfully loaded.")
else:
    print("No database found in Drive. Preparing to build a new one...")

    df = pd.read_csv(csv_file_path)
    df = df.dropna(subset=['section_content'])

    documents = []
    for index, row in df.iterrows():
        act_title = row['act_title']
        section_no = row['section_no']
        content = row['section_content']
        combined_text = f"Act Title: {act_title}\nSection No: {section_no}\nDetails: {content}"

        doc = Document(
            page_content=combined_text,
            metadata={
                "act_title": act_title,
                "section_no": section_no,
                "source": row['source_url'] if pd.notna(row['source_url']) else "Unknown"
            }
        )
        documents.append(doc)

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    docs = text_splitter.split_documents(documents)

    print("Building FAISS vector database. This will be fast if using T4 GPU...")
    vectorstore = FAISS.from_documents(docs, embeddings)

    print(f"Saving vector database to {db_save_path}...")
    vectorstore.save_local(db_save_path)
    print("Database built and saved to Drive!")

# 7. Setup LLM and Authentication
if "GOOGLE_API_KEY" not in os.environ:
    print("\nEnter your Google Gemini API Key:")
    os.environ["GOOGLE_API_KEY"] = getpass()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)

system_prompt = (
    "তুমি 'LawBot', বাংলাদেশের আইনভিত্তিক একটি সহকারী। "
    "তুমি শুধুমাত্র বাংলাদেশ আইন অনুযায়ী তথ্য দেবে। "
    "ব্যবহারকারী যদি কোনো অপরাধ স্বীকার করে বা ক্রাইম সম্পর্কিত প্রশ্ন করে (যেমন চুরি, মারামারি, প্রতারণা), "
    "তাহলে তুমি আইন অনুযায়ী সম্ভাব্য শাস্তি, আইন ধারা এবং সাধারণ আইনি ব্যাখ্যা দেবে। "
    "তুমি কখনোই ব্যবহারকারীকে অপরাধ করতে সাহায্য করবে না বা কিভাবে ধরা এড়ানো যায় তা বলবে না। "
    "সবসময় শান্ত, নিরপেক্ষ এবং শিক্ষামূলকভাবে উত্তর দেবে। "
    "যদি তথ্য না থাকে, বলবে 'প্রদত্ত আইন অনুযায়ী নিশ্চিত তথ্য পাওয়া যায়নি।'\n\n"
    "উত্তর অবশ্যই বাংলা বা ইংরেজিতে হতে পারে (ইউজারের ভাষা অনুযায়ী)।\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 8. Create the RAG Pipeline
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 9. Test the Application
print("\n" + "="*50)
print("LawBot Setup Complete! Running a test query...")
print("="*50)

test_question = "What are the provisions regarding fundamental rights?"
print(f"\nUser Query: {test_question}")

response = rag_chain.invoke({"input": test_question})

print("\nLawBot Answer:")
print("-" * 50)
print(response["answer"])
print("-" * 50)

print("\n[References Used]:")
for doc in response["context"]:
    print(f"- {doc.metadata.get('act_title')} (Section: {doc.metadata.get('section_no')})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 56.3 MB/s eta 0:00:00


/tmp/ipykernel_2140/3233357416.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Mounted at /content/drive
Hardware accelerator set to: CUDA


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Found existing vector database in Drive! Loading instantly...
Database successfully loaded.

Enter your Google Gemini API Key:
··········

LawBot Setup Complete! Running a test query...

User Query: What are the provisions regarding fundamental rights?

LawBot Answer:
--------------------------------------------------
প্রদত্ত তথ্য অনুযায়ী, মৌলিক অধিকার সম্পর্কিত বিধানগুলো মূলত কিছু নির্দিষ্ট আইনের ক্ষেত্রে মৌলিক অধিকারের সাথে অসামঞ্জস্যতা বা সীমাবদ্ধতা থাকা সত্ত্বেও সেগুলোকে বৈধতা প্রদান করে।

সংবিধানের ৪৭ অনুচ্ছেদ অনুযায়ী:

১.  নিম্নোক্ত বিষয়গুলো সম্পর্কিত কোনো আইনকে বাতিল বলে গণ্য করা হবে না, এই যুক্তিতে যে এটি মৌলিক অধিকারের সাথে অসামঞ্জস্যপূর্ণ বা মৌলিক অধিকারকে খর্ব করে:
    *   সম্পত্তির বাধ্যতামূলক অধিগ্রহণ, জাতীয়করণ বা রিকুইজিশন, অথবা এর অস্থায়ী বা স্থায়ী নিয়ন্ত্রণ বা ব্যবস্থাপনা।
    *   বাণিজ্যিক বা অন্যান্য উদ্যোগ পরিচালনাকারী সংস্থাগুলির বাধ্যতামূলক একত্রীকরণ।
    *   এই ধরনের সংস্থাগুলির পরিচালক, ব্যবস্থাপক, এজেন্ট এবং কর্মকর্তাদের অধিকার, অথবা শেয়ার বা স্টক মালিকদের ভোটাধ

In [ ]:
print("\n" + "="*60)
print("LawBot চালু হয়েছে! প্রশ্ন করো (exit লিখলে বন্ধ হবে)")
print("="*60)

while True:
    query = input("\nতোমার প্রশ্ন: ")

    if query.lower() in ["exit", "quit", "stop"]:
        print("LawBot বন্ধ করা হচ্ছে...")
        break

    response = rag_chain.invoke({"input": query})

    print("\n📌 LawBot উত্তর:")
    print("-" * 50)
    print(response["answer"])
    print("-" * 50)


    for doc in response["context"]:
        print(f"- {doc.metadata.get('act_title')} (Section: {doc.metadata.get('section_no')})")


LawBot চালু হয়েছে! প্রশ্ন করো (exit লিখলে বন্ধ হবে)

তোমার প্রশ্ন: exit
LawBot বন্ধ করা হচ্ছে...


In [ ]:
# নয়েজ টেস্টের জন্য ডাইরেক্ট কোয়েরি লিস্ট (কোনো সেভ করার ঝামেলা নেই)
noisy_queries = [
    {
        "original": "What are the provisions regarding fundamental rights?",
        "typo_version": "Whrt r the prvisions regrding fundmentl rghts?", # বানান ভুল
        "distractor_version": "Tell me about fundamental rights, by the way I like playing football." # অপ্রাসঙ্গিক কথা
    },
    {
        "original": "What is the punishment for theft in Bangladesh?",
        "typo_version": "Wat is d punishmnt for thft in Bengladesh?",
        "distractor_version": "Can you explain the punishment for theft? My cat is sleeping right now."
    }
]

print("🔬 Running Robustness to Noise Evaluation...\n")
for q in noisy_queries:
    print(f"Original: {q['original']}")

    # ১. টাইপো টেস্ট
    res_typo = rag_chain.invoke({"input": q["typo_version"]})
    print(f"↳ Typo Input Answer: {res_typo['answer'][:150]}...")

    # ২. ডিস্ট্রাক্টর টেস্ট
    res_dist = rag_chain.invoke({"input": q["distractor_version"]})
    print(f"↳ Distractor Input Answer: {res_dist['answer'][:150]}...\n")

🔬 Running Robustness to Noise Evaluation...

Original: What are the provisions regarding fundamental rights?
↳ Typo Input Answer: আপনার প্রশ্নটি মৌলিক অধিকার (fundamental rights) সংক্রান্ত। তবে, আপনি যে প্রসঙ্গ (context) দিয়েছেন, তাতে মৌলিক অধিকার সম্পর্কে কোনো তথ্য নেই। প্রদত্ত...
↳ Distractor Input Answer: আপনি মৌলিক অধিকার সম্পর্কে জানতে চেয়েছেন। তবে, আপনার প্রদত্ত তথ্যে (ইজমেন্টস অ্যাক্ট, ১৮৮২) মৌলিক অধিকারের বিষয়ে সরাসরি কোনো ধারা বা আলোচনা নেই।

মৌ...

Original: What is the punishment for theft in Bangladesh?
↳ Typo Input Answer: প্রদত্ত তথ্যে বাংলাদেশে চুরির শাস্তির বিষয়ে কোনো বিবরণ নেই। এখানে শুধুমাত্র ১৮১৮ সালের স্টেট প্রিজনার্স রেগুলেশন এবং সংরক্ষিত ও সুরক্ষিত বনাঞ্চলে বিভ...
↳ Distractor Input Answer: বাংলাদেশের আইন অনুযায়ী চুরির শাস্তি নিম্নরূপ:

1.  **সাধারণ চুরি (General Theft) - দণ্ডবিধি, ১৮৬০ এর ধারা ৩৭৯ অনুযায়ী:**
    *   যে ব্যক্তি চুরি করে...



In [ ]:
# সরাসরি তোমার ডাটাবেস থেকে ৩-৪টি ইউনিক আইনের ধারা টেস্টের জন্য নাও
test_cases = [
    {"query": "fundamental rights article 32", "expected_act": "The Constitution", "expected_sec": "32"},
    {"query": "punishment for cheating or fraud", "expected_act": "Penal Code", "expected_sec": "420"}
]

k_values = [1, 3, 5]
retriever_eval = {}

for k in k_values:
    hits = 0
    temp_retriever = vectorstore.as_retriever(search_kwargs={"k": k})

    for case in test_cases:
        docs = temp_retriever.invoke(case["query"])
        # মেটাডাটা চেক
        for doc in docs:
            if case["expected_sec"] in str(doc.metadata.get("section_no")):
                hits += 1
                break

    hit_rate = hits / len(test_cases)
    retriever_eval[f"Hit Rate@{k}"] = hit_rate

print("📊 Retriever-Only Evaluation Summary for Journal Paper:")
print("-" * 50)
for metric, score in retriever_eval.items():
    print(f"{metric}: {score * 100}%")

📊 Retriever-Only Evaluation Summary for Journal Paper:
--------------------------------------------------
Hit Rate@1: 0.0%
Hit Rate@3: 0.0%
Hit Rate@5: 50.0%


In [ ]:
sample_query = "What happens if someone commits cyber defamation in Bangladesh?"

# K=1 বনাম K=3 এর আউটপুট তুলনা
for k in [1, 3]:
    custom_retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    custom_rag_chain = create_retrieval_chain(custom_retriever, question_answer_chain)

    response = custom_rag_chain.invoke({"input": sample_query})
    print(f"=== Results for K = {k} ===")
    print(f"Tokens/Context Chunks Used: {len(response['context'])}")
    print(f"Answer: {response['answer']}\n")

=== Results for K = 1 ===
Tokens/Context Chunks Used: 1
Answer: বাংলাদেশে যদি কেউ সাইবার মানহানি (cyber defamation) করে, তবে তা ডিজিটাল নিরাপত্তা আইন, ২০১৮ (Digital Security Act, 2018) এর অধীনে একটি অপরাধ বলে গণ্য হবে।

এখানে এর পরিণতি সম্পর্কে বিস্তারিত তথ্য দেওয়া হলো:

1.  **আইনের ভিত্তি:**
    *   ডিজিটাল নিরাপত্তা আইন, ২০১৮ এর **ধারা ২৯** সাইবার মানহানি সংক্রান্ত অপরাধ ও শাস্তির বিধান করে।
    *   এই ধারায় বলা হয়েছে যে, যদি কোনো ব্যক্তি ডিজিটাল মাধ্যমে পেনাল কোড, ১৮৬০ (Penal Code, 1860) এর **ধারা ৪৯৯** এ সংজ্ঞায়িত মানহানিকর তথ্য প্রকাশ বা প্রচার করেন, তবে তিনি অপরাধ করেছেন বলে গণ্য হবেন।

2.  **শাস্তি (প্রথমবার অপরাধের জন্য):**
    *   অনধিক **৩ (তিন) বছর** কারাদণ্ড, অথবা
    *   অনধিক **৫ (পাঁচ) লক্ষ টাকা** অর্থদণ্ড, অথবা
    *   উভয় দণ্ড।

3.  **শাস্তি (একই অপরাধ দ্বিতীয়বার বা বারবার করার জন্য):**
    *   অনধিক **৫ (পাঁচ) বছর** কারাদণ্ড, অথবা
    *   অনধিক **১০ (দশ) লক্ষ টাকা** অর্থদণ্ড, অথবা
    *   উভয় দণ্ড।

4.  **অপরাধের প্রকৃতি:**
    *   ডিজিটাল নিরাপত্তা আইন, ২০১৮ এর অধীনে